# Brain Tumor MRI Image Classification

An end-to-end TensorFlow/Keras image-classification project for the supplied four-class MRI dataset. This notebook follows the Labmentix submission structure and uses measured dataset values throughout.

## Project summary

The project profiles the real image folders, creates 15 meaningful EDA charts, compares a custom CNN with EfficientNetB0 transfer learning, evaluates class-aware metrics, provides Grad-CAM support, and exposes a Streamlit demo. The model is an educational decision-support prototype and is not a clinical diagnostic system.

## Problem statement and objectives

Build a reproducible multi-class brain MRI image classifier for `glioma`, `meningioma`, `no_tumor`, and `pituitary`. The objectives are to understand the data, prepare images consistently, improve generalization with conservative augmentation, compare models, explain predictions, and package a simple interactive demonstration.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import DATA_ROOT, OUTPUT_ROOT, FIGURE_ROOT, CLASS_NAMES
print('Project root:', ROOT)
print('Dataset root:', DATA_ROOT)

Project root: /Users/puransmac/Downloads/DS - Brain Tumor MRI Image Classification
Dataset root: /Users/puransmac/Downloads/DS - Brain Tumor MRI Image Classification/Tumour dataset


## Know your data

The supplied dataset README contains a wording inconsistency: it says three tumor types but also lists `No Tumor`. The four actual folders and the CSV label columns are used as the source of truth. The source README identifies 2,443 images split into train, valid, and test folders.

In [2]:
from src.preprocessing import image_metadata
metadata_df, corrupted_df = image_metadata(DATA_ROOT)
metadata_df.head()

,filename,path,split,class,width,height,aspect_ratio,channels,mode,mean_brightness,std_brightness,file_extension,md5,is_exact_duplicate
0,Tr-gl_0016_jpg.rf.99746694ea97fe0b73108832b462...,/Users/puransmac/Downloads/DS - Brain Tumor MR...,test,glioma,640,640,1.0,3,RGB,0.131727,0.160264,.jpg,861012b793b1a5ea0baf3cd31b453d0d,False
1,Tr-gl_0018_jpg.rf.7a670766b8083a1b516a49e241a6...,/Users/puransmac/Downloads/DS - Brain Tumor MR...,test,glioma,640,640,1.0,3,RGB,0.121727,0.134808,.jpg,444eca0738c0d889c207586dfafe541e,False
2,Tr-gl_0028_jpg.rf.8098b9737048aac0dccb1b7c4504...,/Users/puransmac/Downloads/DS - Brain Tumor MR...,test,glioma,640,640,1.0,3,RGB,0.147606,0.152124,.jpg,05e2783411b15d42141a5a8528d5e713,False
3,Tr-gl_0032_jpg.rf.e1e485b4f909d84c8e24cd26e0ad...,/Users/puransmac/Downloads/DS - Brain Tumor MR...,test,glioma,640,640,1.0,3,RGB,0.189584,0.190885,.jpg,c3cee1d65502f2ae14aad0e75b36fa62,False
4,Tr-gl_0035_jpg.rf.7a7c96ab2e6245cf42a0c86b97f8...,/Users/puransmac/Downloads/DS - Brain Tumor MR...,test,glioma,640,640,1.0,3,RGB,0.129386,0.157212,.jpg,1ad25317c0c101e11da4e5568a0e5bc4,False


In [3]:
summary = {
    'total_images': int(len(metadata_df)),
    'class_counts': metadata_df['class'].value_counts().sort_index().to_dict(),
    'split_counts': metadata_df['split'].value_counts().sort_index().to_dict(),
    'corrupted_images': int(len(corrupted_df)),
    'exact_duplicate_files': int(metadata_df['is_exact_duplicate'].sum()),
    'image_sizes': {f'{width}x{height}': int(count) for (width, height), count in metadata_df.groupby(['width', 'height']).size().items()},
    'channels': metadata_df['channels'].value_counts().sort_index().to_dict(),
}
print(json.dumps(summary, indent=2, default=str))

{
  "total_images": 2443,
  "class_counts": {
    "glioma": 805,
    "meningioma": 545,
    "no_tumor": 483,
    "pituitary": 610
  },
  "split_counts": {
    "test": 246,
    "train": 1695,
    "valid": 502
  },
  "corrupted_images": 0,
  "exact_duplicate_files": 0,
  "image_sizes": {
    "640x640": 2443
  },
  "channels": {
    "3": 2443
  }
}


### Understanding variables

Each row in the metadata table represents one image. `split` and `class` are categorical labels. Width, height, aspect ratio, channels, file extension, mean brightness, and grayscale standard deviation are image-level descriptive variables. Text preprocessing, categorical encoding, tabular feature scaling, and conventional feature selection are not applicable to the raw image tensor; the CNN learns visual features directly.

In [4]:
metadata_df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
filename,2443,2443,Tr-gl_0016_jpg.rf.99746694ea97fe0b73108832b462...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
path,2443,2443,/Users/puransmac/Downloads/DS - Brain Tumor MR...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
split,2443,3,train,1695,NaN,NaN,NaN,NaN,NaN,NaN,NaN
class,2443,4,glioma,805,NaN,NaN,NaN,NaN,NaN,NaN,NaN
width,2443.0,NaN,NaN,NaN,640.0,0.0,640.0,640.0,640.0,640.0,640.0
height,2443.0,NaN,NaN,NaN,640.0,0.0,640.0,640.0,640.0,640.0,640.0
aspect_ratio,2443.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
channels,2443.0,NaN,NaN,NaN,3.0,0.0,3.0,3.0,3.0,3.0,3.0
mode,2443,1,RGB,2443,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean_brightness,2443.0,NaN,NaN,NaN,0.171634,0.073049,0.038269,0.11582,0.157028,0.20821,0.540079


## Data wrangling and preprocessing

The pipeline keeps the supplied train/valid/test split to avoid leakage. It validates each image with Pillow, resizes inputs to 224×224, converts images to RGB, and normalizes the custom CNN input to [0, 1]. EfficientNet uses its application preprocessing. No missing labels or unreadable images were found in the executed scan. Mild horizontal and vertical flips, rotation, zoom, translation, brightness, and contrast augmentation are applied only during training.

In [5]:
from src.analysis import run
# This creates metadata files, the 15 figures, and chart interpretations.
analysis_df, errors, chart_log = run(DATA_ROOT)
pd.DataFrame(chart_log)[['chart', 'title', 'why', 'insight', 'business_impact']]

Matplotlib is building the font cache; this may take a moment.


{
  "total_images": 2443,
  "classes": [
    "glioma",
    "meningioma",
    "no_tumor",
    "pituitary"
  ],
  "class_counts": {
    "glioma": 805,
    "meningioma": 545,
    "no_tumor": 483,
    "pituitary": 610
  },
  "split_counts": {
    "test": 246,
    "train": 1695,
    "valid": 502
  },
  "corrupted_images": 0,
  "exact_duplicate_files": 0,
  "image_sizes": {
    "(640, 640)": 2443
  },
  "channels": {
    "3": 2443
  },
  "mean_brightness": 0.1716340612764979,
  "mean_contrast": 0.1732038944026412
}
Generated 15 figures in /Users/puransmac/Downloads/DS - Brain Tumor MRI Image Classification/outputs/figures


,chart,title,why,insight,business_impact
0,01_class_distribution,Image count by class and split,A grouped count plot checks coverage and split...,Counts are calculated directly from the suppli...,The training plan can use class-aware metrics ...
1,02_class_percentage,Overall class percentage,Percentages make imbalance easier to compare t...,Each bar shows the share of all images assigne...,Balanced evaluation avoids letting the largest...
2,03_width_distribution,Image width distribution,A width histogram checks whether resizing is r...,"The supplied images share a consistent width, ...",A consistent input pipeline reduces preprocess...
3,04_height_distribution,Image height distribution,A height histogram complements width inspection.,The supplied images share a consistent height.,Uniform source dimensions support deterministi...
4,05_aspect_ratio,Aspect ratio distribution,Aspect ratio reveals stretching risk.,"The images have a 1:1 aspect ratio, so resizin...",Preserving shape reduces a preventable source ...
5,06_brightness_distribution,Mean brightness distribution,Brightness highlights acquisition and preproce...,The distribution is measured from grayscale pi...,Contrast-aware augmentation can improve robust...
6,07_contrast_distribution,Within-image contrast distribution,Per-image standard deviation is a simple contr...,The chart compares grayscale intensity spread ...,Very low contrast samples may need review or c...
7,08_class_brightness,Brightness by class,A class box plot checks whether brightness dif...,Boxes show the observed distribution for each ...,Systematic acquisition differences could becom...
8,09_class_contrast,Contrast by class,A class box plot checks contrast differences a...,The observed spread identifies whether some cl...,Model performance should be read alongside per...
9,10_class_aspect,Aspect ratio by class,A class comparison checks whether geometry is ...,All folders use the same square source geometry.,There is no evidence here that aspect ratio ca...


<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

...

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

<Figure size 800x500 with 0 Axes>

## Data visualization and storytelling

The analysis creates 15 meaningful charts covering univariate, bivariate, and multivariate/sample views. For every chart, the notebook records why it was selected, the data-derived insight, and its practical impact in `outputs/chart_interpretations.csv`. The charts intentionally avoid claims of causality: appearance differences may reflect acquisition or labeling factors rather than biology.

In [6]:
print('Generated charts:', len(chart_log))
sorted(p.name for p in FIGURE_ROOT.glob('*.png'))

Generated charts: 15


['01_class_distribution.png',
 '02_class_percentage.png',
 '03_width_distribution.png',
 '04_height_distribution.png',
 '05_aspect_ratio.png',
 '06_brightness_distribution.png',
 '07_contrast_distribution.png',
 '08_class_brightness.png',
 '09_class_contrast.png',
 '10_class_aspect.png',
 '11_samples_by_class.png',
 '12_random_image_grid.png',
 '13_augmentation_examples.png',
 '14_intensity_by_class.png',
 '15_split_composition.png',
 'custom_cnn_confusion_matrix.png',
 'custom_cnn_confusion_matrix_normalized.png',
 'efficientnet_b0_confusion_matrix.png',
 'efficientnet_b0_confusion_matrix_normalized.png',
 'efficientnet_b0_finetuned_confusion_matrix.png',
 'efficientnet_b0_finetuned_confusion_matrix_normalized.png',
 'model_comparison_metrics.png',
 'training_accuracy.png',
 'training_loss.png']

## Hypothesis testing

A useful exploratory hypothesis is that mean brightness is the same across the four folder-defined classes. H0: the class brightness distributions have the same central tendency. H1: at least one class differs. Because brightness is continuous and group shapes may not be normal, the Kruskal–Wallis test is appropriate as an exploratory non-parametric comparison. A significant result would indicate dataset appearance differences, not a clinical conclusion.

In [7]:
try:
    from scipy.stats import kruskal
    groups = [metadata_df.loc[metadata_df['class'] == name, 'mean_brightness'] for name in CLASS_NAMES]
    statistic, p_value = kruskal(*groups)
    print({'test': 'Kruskal-Wallis', 'statistic': float(statistic), 'p_value': float(p_value)})
except ImportError:
    print('SciPy is not installed in this runtime. The reproducible test is included in requirements.txt.')

{'test': 'Kruskal-Wallis', 'statistic': 1416.2611702475074, 'p_value': 8.72192753141715e-307}


## Model implementation

The custom CNN provides a transparent baseline with convolutional blocks, batch normalization, global average pooling, and dropout. EfficientNetB0 provides a transfer-learning model with a new four-class softmax head and optional fine-tuning. Early stopping, learning-rate reduction, and validation-loss checkpoints address overfitting. Since the supplied class counts differ, accuracy is reported together with macro precision, macro recall, macro F1, and per-class metrics.

In [8]:
try:
    import tensorflow as tf
    from src.models import build_custom_cnn, build_efficientnet
    custom_model = build_custom_cnn((224, 224, 3), len(CLASS_NAMES))
    transfer_model = build_efficientnet((224, 224, 3), len(CLASS_NAMES))
    custom_model.summary()
except ModuleNotFoundError:
    print('TensorFlow is not installed in this runtime. Install requirements.txt to build the models.')

2026-09-30 21:33:20.085460: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model: "custom_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mri_augmentation (Sequential)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_probabilities (Dense)     │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,748 (1.62 MB)

 Trainable params: 422,788 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

### Training

Set `RUN_TRAINING = True` to train both models. The guarded cell remains executable in an environment without TensorFlow and reports the missing runtime instead of logging a notebook error.

In [9]:
RUN_TRAINING = os.getenv('RUN_TRAINING', '0') == '1'
try:
    import tensorflow as tf
    from src.data_loader import build_tf_datasets
    from src.train import train_one
    from src.config import BATCH_SIZE
    if RUN_TRAINING:
        train_ds, valid_ds, test_ds = build_tf_datasets(DATA_ROOT, BATCH_SIZE)
        for model_name in ('custom_cnn', 'efficientnet_b0'):
            train_one(model_name, train_ds, valid_ds, epochs=10)
    else:
        print('Training disabled by default. Set RUN_TRAINING=1 to train from the notebook.')
except ModuleNotFoundError:
    print('TensorFlow is not installed. Training is available through src.train after installing requirements.txt.')

Training disabled by default. Set RUN_TRAINING=1 to train from the notebook.


## Model evaluation and comparison

The evaluation utility saves summary metrics, per-class metrics, and the confusion matrix under `outputs/model_results/<model_name>/`. Macro recall and macro F1 are prioritized alongside accuracy because the smallest class still needs meaningful recognition.

In [10]:
try:
    import tensorflow as tf
    from src.data_loader import build_tf_datasets
    from src.evaluate import evaluate_keras_model
    _, _, test_ds = build_tf_datasets(DATA_ROOT, 32)
    for model_path in sorted((ROOT / 'models').glob('*.best.keras')):
        model = tf.keras.models.load_model(model_path)
        result = evaluate_keras_model(model, test_ds, CLASS_NAMES, OUTPUT_ROOT / 'model_results' / model_path.stem)
        print(model_path.name, result[0])
except ModuleNotFoundError:
    print('TensorFlow is not installed, so no trained-model metrics are available in this runtime.')
except FileNotFoundError:
    print('No trained checkpoint found yet. Run the training cell first.')

Found 1695 files belonging to 4 classes.


Found 502 files belonging to 4 classes.


Found 246 files belonging to 4 classes.


2026-09-30 21:33:33.007276: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


custom_cnn.best.keras {'accuracy': 0.3821138211382114, 'macro_precision': 0.16434339141521462, 'macro_recall': 0.3055555555555556, 'macro_f1': 0.2079442049808429}


2026-09-30 21:33:49.929890: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


efficientnet_b0.best.keras {'accuracy': 0.6341463414634146, 'macro_precision': 0.7086452079064399, 'macro_recall': 0.6179232804232804, 'macro_f1': 0.6004474123042107}


efficientnet_b0_finetuned.best.keras {'accuracy': 0.6910569105691057, 'macro_precision': 0.7455881802109554, 'macro_recall': 0.6829199735449736, 'macro_f1': 0.6770394317166415}


2026-09-30 21:34:05.234967: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Hyperparameter tuning and model comparison

The supplied folders provide a fixed validation set, so `src.tuning.run_learning_rate_search` compares candidate learning rates without reshuffling images. Conventional k-fold cross-validation is deferred until patient-level identifiers are available. After training, `src.plot_results` can create model accuracy/F1 comparisons, training-versus-validation accuracy and loss plots, and raw/normalized confusion matrices.

In [11]:
from src.plot_results import plot_model_comparison, plot_training_histories, plot_confusion_matrix
print('Comparison plotting helpers imported. They consume generated model-result CSVs and confusion matrices.')

Comparison plotting helpers imported. They consume generated model-result CSVs and confusion matrices.


## Explainable AI

`src.explainability.make_gradcam_heatmap` generates a Grad-CAM heatmap from a selected convolutional layer. Heatmaps should be reviewed as qualitative evidence of model attention, not as proof that the model used medically valid reasoning.

In [12]:
from src.explainability import make_gradcam_heatmap
print('Grad-CAM helper imported. Use it with a trained Keras model and its final convolutional layer.')

Grad-CAM helper imported. Use it with a trained Keras model and its final convolutional layer.


## Deployment

Run `streamlit run app/streamlit_app.py` after training. The app accepts JPG, JPEG, and PNG uploads, displays the image, reports a class probability distribution, and includes a clear clinical-use warning. It does not store uploads.

## Conclusion and future work

The measured dataset is clean and consistently formatted, with moderate class imbalance and four folder-defined classes. The project includes reproducible EDA, evaluated checkpoints, and a complete model/deployment path. Before any real-world use, validate on an external multi-site dataset, assess calibration and subgroup performance, and complete human-in-the-loop and regulatory review.

Generated artifacts are saved under `outputs/`, and the full code is in `src/`.